In [ ]:
3 import pandas as pd
import numpy as np

df = pd.read_csv('Sample - Superstore.csv',encoding="latin1")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [ ]:
df.columns

Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State',
       'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category',
       'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit'],
      dtype='object')

In [ ]:
df.columns = df.columns.str.strip().str.lower()
df.columns

Index(['row id', 'order id', 'order date', 'ship date', 'ship mode',
       'customer id', 'customer name', 'segment', 'country', 'city', 'state',
       'postal code', 'region', 'product id', 'category', 'sub-category',
       'product name', 'sales', 'quantity', 'discount', 'profit'],
      dtype='object')

In [ ]:
df['product name'].unique()

array(['Bush Somerset Collection Bookcase',
       'Hon Deluxe Fabric Upholstered Stacking Chairs, Rounded Back',
       'Self-Adhesive Address Labels for Typewriters by Universal', ...,
       'Eureka Hand Vacuum, Bagless', 'LG G2',
       'Eldon Jumbo ProFile Portable File Boxes Graphite/Black'],
      dtype=object)

In [ ]:
df['order date'] = pd.to_datetime(df['order date'], format='%m/%d/%Y')
df['ship date']  = pd.to_datetime(df['ship date'],  format='%m/%d/%Y')

print(df['order date'].dtype)
print(df['order date'].head(3))

datetime64[ns]
0   2016-11-08
1   2016-11-08
2   2016-06-12
Name: order date, dtype: datetime64[ns]


In [ ]:
df['year']          = df['order date'].dt.year
df['month']         = df['order date'].dt.month
df['yearmonth']     = df['order date'].dt.to_period('M').astype(str)
df['ship_days']      = (df['ship date'] - df['order date']).dt.days
df['profit_margin']  = df['profit'] / df['sales']

print(df[['order date', 'year', 'month', 'yearmonth', 'ship_days', 'profit_margin']].head(4))


  order date  year  month yearmonth  ship_days  profit_margin
0 2016-11-08  2016     11   2016-11          3           0.16
1 2016-11-08  2016     11   2016-11          3           0.30
2 2016-06-12  2016      6   2016-06          4           0.47
3 2015-10-11  2015     10   2015-10          7          -0.40


In [ ]:
df['IsLoss']       = (df['profit'] < 0).astype(int)
df['HighDiscount'] = (df['discount'] >= 0.5).astype(int)

print(f"Loss rows:          {df['IsLoss'].sum():,}")
print(f"High discount rows: {df['HighDiscount'].sum():,}")
print(f"\nSample loss rows:")
print(df[df['IsLoss'] == 1][['product name', 'sales', 'discount', 'profit']].head(4))

Loss rows:          1,871
High discount rows: 922

Sample loss rows:
                                         product name     sales  discount  \
3       Bretford CR4500 Series Slim Rectangular Table  957.5775      0.45   
14  Holmes Replacement Filter for HEPA Air Cleaner...   68.8100      0.80   
15   Storex DuraTech Recycled Plastic Frosted Binders    2.5440      0.80   
23                 Global Deluxe Stacking Chair, Gray   71.3720      0.30   

      profit  
3  -383.0310  
14 -123.8580  
15   -3.8160  
23   -1.0196  


In [ ]:
loss_with_disc = df[(df['IsLoss']==1) & (df['HighDiscount']==1)]
print(f"\nLoss rows that also had high discount: {len(loss_with_disc):,}")
print(f"That's {len(loss_with_disc)/df['IsLoss'].sum()*100:.1f}% of all losses")


Loss rows that also had high discount: 922
That's 49.3% of all losses


In [ ]:
str_cols = df.select_dtypes(include='object').columns

for col in str_cols:
    df[col] = df[col].str.strip()

for col in ['region', 'category', 'sub-category', 'segment', 'ship mode']:
    df[col] = df[col].str.title()

print(df[['region', 'category', 'sub-category', 'segment', 'ship mode']].head())

  region         category sub-category    segment       ship mode
0  South        Furniture    Bookcases   Consumer    Second Class
1  South        Furniture       Chairs   Consumer    Second Class
2   West  Office Supplies       Labels  Corporate    Second Class
3  South        Furniture       Tables   Consumer  Standard Class
4  South  Office Supplies      Storage   Consumer  Standard Class


In [ ]:
df.to_csv('superstore_cleaned.csv', index=False)
print("Saved superstore_cleaned.csv")

Saved superstore_cleaned.csv
